# H05 — In class: signal processing on real data

> **References**
>
> (Orfanidis 1996), (Harris 1978), (Chatterji et al. 2004), (Torrence and Compo 1998), (LIGO Scientific and Virgo Collaborations 2020), (Zevin et al. 2017), (LIGO Scientific Collaboration and Virgo Collaboration 2021), (LIGO Scientific, Virgo and KAGRA Collaborations 2023)

## What we show here

We take the tools of lecture 12 and use them on real detector data, in the order of the lecture. All the code is in view. We run it cell by cell and look at what comes out.

**The data.**
- 32 s of Virgo strain (V1) from the third observing run, sampled at 4096 Hz. It is the record of `H04`: the file `labs/data/gw190814_V1.hdf5`. It is centered on the event GW190814. The signal-to-noise ratio of this event is very different in the three detectors. For what we do here the Virgo record is detector noise.
- 32 s of LIGO Hanford strain (H1) from the first observing run, around a **blip**: a short glitch. The file is `labs/data/Blip_GM7j3JNHDv_H1.hdf5`. The class of the glitch comes from Gravity Spy.
- Both records come from the Gravitational Wave Open Science Center (GWOSC, CC BY 4.0). We download them live, as in `H04` at home. A copy of each is in `labs/data/`, for when the network is down.

**What we do.** First we download the two records from GWOSC. Then:
1. The record and its sampling frequency.
2. Its spectrum, before any filter.
3. A digital low-pass: its coefficients, its poles and its gain.
4. Keep one sample in $M$, without and with the low-pass: aliasing in real data.
5. A band-pass: the record in time and in frequency, before and after.
6. A notch on one line.
7. A glitch in time.
8. The same glitch in a spectrogram, with a short and a long window.
9. The same glitch in the Q transform.
10. The same glitch with the Morlet wavelet.

> **The symbols**
>
> - **$x[n]$** — the time series: the strain at the sample $n$, $x[n] = x(n\,\Delta t)$
> - **$f_{s}$**, **$\Delta t = 1/f_{s}$** — the sampling frequency (Hz) and the sampling interval (s)
> - **$N$**, **$T = N\,\Delta t$** — the number of samples and the duration of the record (s)
> - **$f_{N} = f_{s}/2$** — the Nyquist frequency (Hz)
> - **$\sqrt{S}$** — the amplitude spectral density (ASD), in Hz$^{-1/2}$, from Welch's method as in `H04`
> - **$M$** — we keep one sample in $M$; the new rate is $f_{s}/M$
> - **$f_{c}$**, **$W_{n} = f_{c}/f_{N}$** — the cutoff frequency (Hz) and the normalized cutoff that the design function takes
> - **$b_{k}$**, **$a_{k}$** — the coefficients of the filter; **$G(z) = B(z)/A(z)$** — its transfer function; **$|G|$** — the gain
> - **$r$**, **$\Delta f$** — the radius of the poles of the notch and the width of the notch (Hz)
> - **$\sigma$** — the standard deviation of the filtered noise, away from the glitch
> - **$N_{w}$** — the number of samples in one window of the spectrogram
> - **$Q$** — the quality factor of a tile, its center frequency over its width
> - **$a$**, **$\omega_{0}$** — the scale of the wavelet (s) and the parameter of the Morlet wavelet, $\omega_{0} = 6$
> - **$\psi$**, **$\psi_{a,b}$** — the mother wavelet and its copy at the scale $a$ and the shift $b$ (s); **$W(a,b)$** — the wavelet transform; **$|W|^{2}$** — the scalogram
> - **$\sigma_{t}$**, **$\sigma_{f}$** — the widths of a tile in time (s) and in frequency (Hz)

The order of the demonstration: the record, then the filters, then the time–frequency plane.

![The record and its spectrum, then a low-pass and one sample in M, then a band-pass and a notch, then a glitch in the spectrogram and the Q transform, then the wavelets.](figures/h05class_workflow.png)

**The first cell.** It loads the libraries and the course colors. Among the libraries is `gwpy`, which downloads data from GWOSC. It defines two functions that read the two kinds of file in `labs/data/`. A third function gives the ASD with Welch's method, as in `H04`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import h5py
import sys, os
import warnings
from concurrent.futures import ThreadPoolExecutor
# gwpy loads a library that prints a long notice in a notebook; we silence it
warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')
from gwpy.timeseries import TimeSeries

# the course root holds gwcourse_palette.py; find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    if os.path.dirname(here) == here:
        raise FileNotFoundError('gwcourse_palette.py not found: open the notebook from the course folder')
    here = os.path.dirname(here)
sys.path.insert(0, here)
sys.path.insert(0, os.path.join(here, 'labs'))

from gwcourse_palette import P, apply_style
from gwcourse_signal import q_transform
apply_style()

# the copies in labs/data: the Virgo record of H04 and a blip in LIGO Hanford
VIRGO = os.path.join(here, 'labs', 'data', 'gw190814_V1.hdf5')
BLIP = os.path.join(here, 'labs', 'data', 'Blip_GM7j3JNHDv_H1.hdf5')


def read_gwosc(path):
    """Read one GWOSC strain file.

    Parameters
    ----------
    path : str, the hdf5 file

    Returns
    -------
    (strain, times, fs) : the array x[n]; the GPS time of each sample (s); the sampling frequency (Hz)
    """
    # the strain, the GPS time of the first sample and the time step
    with h5py.File(path, 'r') as f:
        strain = f['strain/Strain'][:]
        gps_start = float(f['meta/GPSstart'][()])
        dt = float(f['strain/Strain'].attrs['Xspacing'])
    fs = 1.0 / dt
    times = gps_start + np.arange(strain.size) * dt
    record = (strain, times, fs)
    return record


def read_glitch(path):
    """Read one glitch file of labs/data: one dataset, with its start time x0 and its time step dx.

    Parameters
    ----------
    path : str, the hdf5 file

    Returns
    -------
    (strain, times, fs) : the array x[n]; the GPS time of each sample (s); the sampling frequency (Hz)
    """
    # the only dataset of the file, named after the channel
    with h5py.File(path, 'r') as f:
        data = f[next(iter(f.keys()))]
        strain = data[()].astype(float)
        gps_start = float(data.attrs['x0'])
        dt = float(data.attrs['dx'])
    fs = 1.0 / dt
    times = gps_start + np.arange(strain.size) * dt
    record = (strain, times, fs)
    return record


def welch_asd(x, fs, seconds):
    """Return the one-sided ASD by Welch's method: Hann window, 50 percent overlap.

    Parameters
    ----------
    x       : array, the record
    fs      : float, sampling frequency (Hz)
    seconds : float, the length of a segment (s)

    Returns
    -------
    (freq, asd) : the frequencies (Hz); the amplitude spectral density (Hz^-1/2)
    """
    # the PSD of Welch's method, then its square root
    freq, psd = signal.welch(x, fs=fs, nperseg=int(seconds * fs), window='hann')
    asd = np.sqrt(psd)
    curve = (freq, asd)
    return curve

## The data from GWOSC

**GWOSC.** The Gravitational Wave Open Science Center, <https://gwosc.org>, publishes the strain of LIGO, Virgo and KAGRA for everybody to use (CC BY 4.0). The data are described in (LIGO Scientific Collaboration and Virgo Collaboration 2021) for the first two observing runs and in (LIGO Scientific, Virgo and KAGRA Collaborations 2023) for the third.

**How a stretch is chosen.** Three things say which data we want.
- The **detector**: `'H1'` is LIGO Hanford, `'L1'` LIGO Livingston, `'V1'` Virgo.
- The **GPS start and end** of the stretch, in seconds. GPS time counts the seconds from 6 January 1980.
- The **sample rate**: 4096 Hz.

**The download.** `TimeSeries.fetch_open_data` from the package `gwpy` asks GWOSC for the stretch and returns it as a `TimeSeries`. It holds the strain samples (`.value`), the sample rate (`.sample_rate`), the GPS time of the first sample (`.t0`) and the duration (`.duration`).

**If the network is down.** Two functions do the work. `start_download` sends the request. The download then runs in the background. `fetch_or_read` waits for it at most `max_wait` seconds. If nothing comes back in time, it says so in one line and reads the copy in `labs/data/`. The steps below run in the same way in both cases.

In [ ]:
# the downloads run in the background, so that the notebook is free while they arrive
downloads = ThreadPoolExecutor(max_workers=2)


def start_download(detector, gps_start, gps_end, sample_rate):
    """Ask GWOSC for one stretch of strain; the download runs in the background.

    Parameters
    ----------
    detector    : str, 'H1', 'L1' or 'V1'
    gps_start   : float, the GPS time of the first sample (s)
    gps_end     : float, the GPS time of the end of the stretch (s)
    sample_rate : float, samples per second (Hz)

    Returns
    -------
    request : the running download; fetch_or_read collects it
    """
    # the same call as in H04; timeout: the longest silence of the server we accept (s)
    request = downloads.submit(TimeSeries.fetch_open_data, detector, gps_start, gps_end,
                               sample_rate=sample_rate, timeout=20)
    return request


def fetch_or_read(request, cached_file, read_cached, max_wait):
    """Wait for a download at most max_wait seconds; if it fails, read the copy on disk.

    Parameters
    ----------
    request     : the download started by start_download
    cached_file : str, the copy of the same record in labs/data
    read_cached : the function that reads that kind of file (read_gwosc or read_glitch)
    max_wait    : float, the longest wait for GWOSC (s)

    Returns
    -------
    (strain, times, fs) : the array x[n]; the GPS time of each sample (s); the sampling frequency (Hz)
    """
    try:
        # the TimeSeries from GWOSC, if it arrives within max_wait seconds
        data = request.result(timeout=max_wait)
        strain = data.value
        times = data.times.value
        fs = data.sample_rate.value
        print('downloaded from GWOSC')
    except Exception as error:
        # a slow server, no network or a missing stretch: we read the copy instead
        if isinstance(error, TimeoutError):
            reason = f'nothing from GWOSC after {max_wait:.0f} s'
        else:
            reason = 'GWOSC cannot be reached'
        strain, times, fs = read_cached(cached_file)
        print(f'{reason}: we read the copy {os.path.basename(cached_file)}')
    record = (strain, times, fs)
    return record

**The Virgo record.** 32 s of Virgo strain from the third observing run, the record of `H04`. It contains the event GW190814. We write the choice as four names, so that we can change them and run the cell again.

In [ ]:
# the stretch: the detector, the GPS start and end, the sample rate
detector = 'V1'
gps_start = 1249852241          # 14 August 2019, third observing run
gps_end = gps_start + 32        # 32 s of data
sample_rate = 4096              # samples per second (Hz)

# ask GWOSC, wait at most 30 s, then work with what came back
virgo_request = start_download(detector, gps_start, gps_end, sample_rate)
strain, times, fs = fetch_or_read(virgo_request, VIRGO, read_gwosc, max_wait=30)

# what came back
print(f'{detector}: GPS start {times[0]:.0f}, {strain.size / fs:.0f} s at {fs:.0f} Hz, '
      f'{strain.size} samples')

**What to look at.** 131072 samples, 32 s at 4096 Hz: the same numbers as the file of `H04`. From here on the Virgo record is `strain`, with its GPS times `times` and its sampling frequency `fs`.

**The Hanford record.** 32 s of LIGO Hanford strain from the first observing run, with the blip 16 s after the start. GWOSC keeps the data of that run in files of 4096 s, about 130 MB each. The download needs the whole file, so it takes a few minutes. We ask for it now. It arrives in the background while we work on Virgo. We collect it at step 7.

In [ ]:
# the stretch: 16 s before the peak of the blip, on a sample of the 4096 Hz grid
blip_detector = 'H1'
blip_start = 1132159639.4655762       # 21 November 2015, first observing run
blip_end = blip_start + 32            # 32 s of data
blip_request = start_download(blip_detector, blip_start, blip_end, sample_rate)
print(f'{blip_detector}: download from GPS {blip_start:.3f} started')

## 1 · The record and its sampling frequency

**What we do.** The ADC of the detector reads the strain every $\Delta t$ seconds and keeps the numbers: $x[n] = x(n\,\Delta t)$. From $f_{s}$ we get everything else. The Nyquist frequency $f_{N} = f_{s}/2$ is the highest frequency the record can show. The bins of the DFT of the whole record are $f_{s}/N = 1/T$ apart.

We draw 20 ms of the record, with each sample as a dot.

In [ ]:
# the numbers of the sampling
dt = 1 / fs                    # the sampling interval (s)
n_samples = strain.size        # N
duration = n_samples * dt      # T = N dt (s)
f_nyquist = fs / 2             # f_N (Hz)
bin_step = fs / n_samples      # the step of the DFT bins, f_s / N = 1 / T (Hz)
print(f"f_s = {fs:.0f} Hz, dt = {dt * 1e3:.4f} ms, N = {n_samples}, T = {duration:.0f} s")
print(f"f_N = {f_nyquist:.0f} Hz, bins of the DFT {bin_step:.5f} Hz apart")

# the fast ripple of H04 is the violin mode at 445.25 Hz: how many samples in one period?
f_violin = 445.25
print(f"samples in one period of {f_violin} Hz: f_s / f = {fs / f_violin:.1f}")

# 20 ms of the record, from 10 s after the start: the samples as dots on a thin line
first = int(10.0 * fs)
n_show = int(0.020 * fs)
t_show = np.arange(n_show) * dt * 1e3
x_show = strain[first:first + n_show] * 1e19
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(t_show, x_show, color=P['GREY_MID'], lw=0.8)
ax.plot(t_show, x_show, 'o', color=P['BRICK'], ms=3.5, label='the samples $x[n]$')
ax.set(xlabel='time (ms)', ylabel='Virgo strain ($10^{-19}$)')
ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()

**What to look at.** The ripple is the violin mode of `H04`. It has about nine samples in each period, more than two, so the record shows it. The record can show nothing above 2048 Hz.

## 2 · The spectrum before any filter

**What we do.** We estimate the ASD with Welch's method: segments of 4 s, a Hann window, the squared DFT averaged over the segments. This is the picture of `H04`. We mark two lines of that laboratory. The line at 150 Hz is a multiple of the mains. The line at 445.25 Hz is a violin mode. The filters of the next steps act on this spectrum.

In [ ]:
# the ASD of the whole record, segments of 4 s
freq, asd = welch_asd(strain, fs, 4.0)

# the two lines, and the median of the ASD within 25 Hz of each: the continuum under the line
lines = (150.0, 445.25)
for f_line in lines:
    i = np.argmin(np.abs(freq - f_line))
    near = np.abs(freq - f_line) < 25
    print(f"{freq[i]:7.2f} Hz: ASD {asd[i]:.1e}, continuum {np.median(asd[near]):.1e}, ratio {asd[i] / np.median(asd[near]):.0f}")

# the ASD on log axes, with a dot on each line
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.loglog(freq[1:], asd[1:], color=P['BRICK'], lw=0.9, label='Virgo, O3')
for f_line in lines:
    i = np.argmin(np.abs(freq - f_line))
    ax.plot(freq[i], asd[i] * 1.6, 'v', color=P['INK'], ms=7)
ax.plot([], [], 'v', color=P['INK'], ms=7, label='the lines at 150 Hz and 445.25 Hz')
ax.set(xlim=(10, f_nyquist), ylim=(1e-24, 1e-18), xlabel='frequency (Hz)', ylabel=r'$\sqrt{S}$  (Hz$^{-1/2}$)')
ax.legend(frameon=False, loc='upper right')
plt.show()

**What to look at.** The wall at low frequency, the bucket and the narrow lines. The line at 150 Hz is about 700 times above the continuum near it. The violin mode at 445.25 Hz is about 1900 times above. Keep this picture: every filter below changes it.

## 3 · A digital low-pass: coefficients, poles and gain

**What we do.** We do not compute the coefficients by hand. We give three choices to a design function: the sampling frequency, the cutoff and the order. It returns the coefficients $b_{k}$ and $a_{k}$. The cutoff goes in as $W_{n} = f_{c}/f_{N}$. We use `signal.butter` of SciPy with $f_{c} = 200$ Hz and order 8.

**The checks.** The filter has feedback coefficients $a_{k}$, so it is an IIR. It is stable if every pole of $G(z)$ is inside the unit circle. On the unit circle, $G$ gives the gain at each frequency. Applied forwards and backwards, the gain is $|G|^{2}$.

In [ ]:
# the design: sampling frequency, cutoff and order in; the coefficients out
f_cut = 200.0          # f_c (Hz)
order = 8
w_n = f_cut / f_nyquist
b_low, a_low = signal.butter(order, w_n, btype='low')
print(f"W_n = {w_n:.4f}; {b_low.size} coefficients b_k and {a_low.size} coefficients a_k: an IIR")

# the poles are the roots of A(z); stable if all of them are inside the unit circle
poles_low = np.roots(a_low)
print(f"largest |pole| = {np.abs(poles_low).max():.4f}")

# the gain on the unit circle: once, and twice for forwards and backwards
f_gain, g_low = signal.freqz(b_low, a_low, worN=8192, fs=fs)
gain_low = np.abs(g_low)
for f_check in (100.0, 200.0, 256.0, 445.25):
    i = np.argmin(np.abs(f_gain - f_check))
    print(f"{f_check:7.2f} Hz: |G| = {gain_low[i]:.1e}, |G|^2 = {gain_low[i] ** 2:.1e}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(f_gain, gain_low, color=P['BLUE'], lw=1.6, label='$|G|$, once')
ax.semilogy(f_gain, gain_low ** 2, color=P['BRICK'], lw=1.6, ls='--', label='$|G|^2$, forwards and backwards')
ax.axvline(f_cut, color=P['GREY_MID'], lw=0.8, ls=':')
ax.set(xlim=(0, 600), ylim=(1e-8, 3), xlabel='frequency (Hz)', ylabel='gain')
ax.legend(frameon=False, loc='lower left')
plt.show()

**What to look at.** Every pole is inside the unit circle: the filter is stable. Below 100 Hz the gain is 1. At the violin mode the gain is about $10^{-3}$ once and $10^{-6}$ twice.

## 4 · One sample in $M$: aliasing in real data

**What we do.** We keep one sample in $M = 8$. The new rate is $f_{s}/M = 512$ Hz and the new Nyquist frequency is 256 Hz. The violin mode at 445.25 Hz is above it. It must appear at its alias:

$$
f_{\mathrm{alias}} = \left|\,f - k\,f_{s}'\right|, \qquad k \text{ the integer closest to } f/f_{s}',
$$

with $f_{s}' = f_{s}/M$ the new rate. We do it twice: once by keeping the samples as they are and once after the low-pass of step 3. Keeping fewer samples is a new sampling, so it needs its own low-pass first.

In [ ]:
# the new rate and the alias of the violin mode
M = 8
fs_new = fs / M
k = round(f_violin / fs_new)
f_alias = abs(f_violin - k * fs_new)
print(f"new rate {fs_new:.0f} Hz, new f_N {fs_new / 2:.0f} Hz; {f_violin} Hz folds to {f_alias:.2f} Hz")

# one sample in M: without and with the low-pass forwards and backwards before
plain = strain[::M]
low_passed = signal.filtfilt(b_low, a_low, strain)[::M]
freq_new, asd_plain = welch_asd(plain, fs_new, 4.0)
freq_new, asd_low = welch_asd(low_passed, fs_new, 4.0)

# the ASD at the alias and the median between 100 and 200 Hz, for the three records
i = np.argmin(np.abs(freq - f_alias))
j = np.argmin(np.abs(freq_new - f_alias))
mid, mid_new = (freq > 100) & (freq < 200), (freq_new > 100) & (freq_new < 200)
print(f"at {f_alias:.2f} Hz:  4096 Hz {asd[i]:.1e}   one in M {asd_plain[j]:.1e}   low-pass, then one in M {asd_low[j]:.1e}")
print(f"median 100-200 Hz: 4096 Hz {np.median(asd[mid]):.1e}   one in M {np.median(asd_plain[mid_new]):.1e}   low-pass, then one in M {np.median(asd_low[mid_new]):.1e}")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.loglog(freq[1:], asd[1:], color=P['GREY_MID'], lw=2.4, label='4096 Hz, as recorded')
ax.loglog(freq_new[1:], asd_plain[1:], color=P['BRICK'], lw=0.9, label='one sample in 8')
ax.loglog(freq_new[1:], asd_low[1:], color=P['BLUE'], lw=0.9, label='low-pass, then one sample in 8')
ax.set(xlim=(10, fs_new / 2), ylim=(1e-24, 1e-18), xlabel='frequency (Hz)', ylabel=r'$\sqrt{S}$  (Hz$^{-1/2}$)')
ax.legend(frameon=False, loc='upper right')
plt.show()

**What to look at.** Without the low-pass a strong line appears at 66.75 Hz, where the recorded data have none. Nothing in the new record says that it is a violin mode. The noise above 256 Hz has also folded into the band and raised the floor. With the low-pass first, the blue curve follows the gray one.

## 5 · A band-pass: the record in time and in frequency

**What we do.** A band-pass keeps the frequencies between two corners and removes the rest. We keep 30 to 300 Hz: above the wall and below the violin modes. It is an IIR of order 4 from the same design function. We apply it forwards and backwards with `signal.filtfilt`: zero phase, so no delay. The first and the last samples are corrupted by the start of the filter, so we drop 2 s at each end.

In [ ]:
# the band-pass: the two corners in units of f_N
band = (30.0, 300.0)
b_band, a_band = signal.butter(4, [band[0] / f_nyquist, band[1] / f_nyquist], btype='band')
print(f"largest |pole| = {np.abs(np.roots(a_band)).max():.4f}")

# forwards and backwards, then drop 2 s at each end
edge = int(2.0 * fs)
band_passed = signal.filtfilt(b_band, a_band, strain)
inner = slice(edge, strain.size - edge)
print(f"rms before {strain[inner].std():.2e}, after {band_passed[inner].std():.2e}")

# time: the same 0.5 s before and after; frequency: the two ASDs
t_rel = times - times[0]
show = (t_rel > 10) & (t_rel < 10.5)
freq_bp, asd_bp = welch_asd(band_passed[inner], fs, 4.0)
for f_line in lines:
    i = np.argmin(np.abs(freq - f_line))
    print(f"ASD at {f_line} Hz: before {asd[i]:.1e}, after {asd_bp[i]:.1e}")
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2), gridspec_kw={'width_ratios': [1, 1, 1.3]})
ax[0].plot(t_rel[show], strain[show] * 1e19, color=P['BRICK'], lw=0.6)
ax[0].set(xlabel='time (s)', ylabel='strain ($10^{-19}$)', title='before')
ax[1].plot(t_rel[show], band_passed[show] * 1e21, color=P['BLUE'], lw=0.6)
ax[1].set(xlabel='time (s)', ylabel='strain ($10^{-21}$)', title='after, 30 to 300 Hz')
ax[2].loglog(freq[1:], asd[1:], color=P['BRICK'], lw=0.9, label='before')
ax[2].loglog(freq_bp[1:], asd_bp[1:], color=P['BLUE'], lw=0.9, label='after')
ax[2].set(xlim=(10, f_nyquist), ylim=(1e-27, 1e-18), xlabel='frequency (Hz)', ylabel=r'$\sqrt{S}$  (Hz$^{-1/2}$)')
ax[2].legend(frameon=False, loc='upper right')
plt.tight_layout()
plt.show()

**What to look at.** In time, the slow wave and the fast ripple are gone. Note the two vertical scales: the rms falls by a factor of about fifteen. What is left looks like one oscillation. In frequency, the band from 30 to 300 Hz is unchanged and both ends fall. The violin mode is about 45 times lower, but it is still above the floor. The line at 150 Hz is inside the band, so it does not change. The oscillation in time is this line.

## 6 · A notch on one line

**What we do.** A notch removes one narrow line and leaves the rest of the band as it is. We build it as the lecture says. Two zeros on the unit circle at the angle $\theta = 2\pi f/f_{s}$ of the line: there the gain is zero. Two poles just inside, at the same angle and at the radius $r$: just off the line the gain is one again. The width is $\Delta f \simeq (1 - r)\,f_{s}/\pi$, so for a width $\Delta f$ we take

$$
r = 1 - \frac{\pi\,\Delta f}{f_{s}}.
$$

`np.poly` turns the zeros into the coefficients of $B(z)$ and the poles into those of $A(z)$. We notch the line at 150 Hz in the band-passed record, with $\Delta f = 1$ Hz.

In [ ]:
def notch_coefficients(f_line, delta_f, fs):
    """The coefficients of a notch: zeros on the unit circle at the line, poles at the radius r.

    Parameters
    ----------
    f_line  : float, the frequency of the line (Hz)
    delta_f : float, the width of the notch (Hz)
    fs      : float, sampling frequency (Hz)

    Returns
    -------
    (b, a, r) : the coefficients of B(z) and A(z); the radius of the poles
    """
    # the angle of the line on the unit circle and the radius of the poles
    theta = 2 * np.pi * f_line / fs
    r = 1 - np.pi * delta_f / fs
    zeros = np.array([np.exp(1j * theta), np.exp(-1j * theta)])
    # B(z) has the zeros, A(z) has the poles r e^{+-i theta}; the coefficients are real
    b = np.real(np.poly(zeros))
    a = np.real(np.poly(r * zeros))
    notch = (b, a, r)
    return notch


# the notch at 150 Hz, 1 Hz wide
f_mains = 150.0
b_notch, a_notch, r = notch_coefficients(f_mains, 1.0, fs)
print(f"r = {r:.5f}; b_k = {np.round(b_notch, 4)}; a_k = {np.round(a_notch, 4)}")
f_check = np.array([140.0, 149.5, 150.0, 150.5, 160.0])
g_check = np.abs(signal.freqz(b_notch, a_notch, worN=f_check, fs=fs)[1])
print("|G| at " + ", ".join(f"{f:.1f} Hz: {g:.2f}" for f, g in zip(f_check, g_check)))

# the band-passed record through the notch, forwards and backwards
notched = signal.filtfilt(b_notch, a_notch, band_passed)
freq_n, asd_n = welch_asd(notched[inner], fs, 4.0)
i = np.argmin(np.abs(freq_bp - f_mains))
rms_bp, rms_n = band_passed[inner].std(), notched[inner].std()
print(f"ASD at 150 Hz: before {asd_bp[i]:.1e}, after {asd_n[i]:.1e}")
print(f"rms: before {rms_bp:.3e}, after {rms_n:.3e}; the line had {100 * (1 - (rms_n / rms_bp) ** 2):.0f} % of the power of the band")

# time: the same 0.5 s as in step 5, before and after the notch; frequency: 15 Hz on each side of the line
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2), gridspec_kw={'width_ratios': [1, 1, 1.3]})
ax[0].plot(t_rel[show], band_passed[show] * 1e21, color=P['BLUE'], lw=0.6)
ax[0].set(xlabel='time (s)', ylabel='strain ($10^{-21}$)', title='band-pass')
ax[1].plot(t_rel[show], notched[show] * 1e21, color=P['SAGE_DARK'], lw=0.6)
ax[1].set(xlabel='time (s)', ylabel='strain ($10^{-21}$)', title='band-pass and notch')
zoom = (freq_bp > 135) & (freq_bp < 165)
ax[2].semilogy(freq_bp[zoom], asd_bp[zoom], color=P['BLUE'], lw=1.2, label='band-pass')
ax[2].semilogy(freq_n[zoom], asd_n[zoom], color=P['SAGE_DARK'], lw=1.2, label='band-pass and notch')
ax[2].set(xlabel='frequency (Hz)', ylabel=r'$\sqrt{S}$  (Hz$^{-1/2}$)')
ax[2].legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.show()

**What to look at.** The gain is zero at 150 Hz, about 0.7 half a hertz away and one 10 Hz away. In the ASD the line is gone and the continuum around it does not change. In time, the oscillation is gone and what is left looks like noise. This one line had 96 percent of the power of the band: the rms falls by a factor of about five.

## 7 · A glitch in time

**What we do.** Now a stretch that has something short in it: the blip in LIGO Hanford. We collect the record we asked GWOSC for at the start. Its peak is 16 s after the start of the record. The spectrum of the whole record cannot say when it happened. So we look in time first. We band-pass 40 to 300 Hz and notch the mains, which is 60 Hz in the USA. Then we draw one second in units of $\sigma$, the standard deviation of the filtered noise between 1 and 10 s from the glitch.

In [ ]:
# the blip record from GWOSC, or its copy; times from its peak, 16 s after the start
blip, blip_times, fs_blip = fetch_or_read(blip_request, BLIP, read_glitch, max_wait=60)
t_blip = blip_times - blip_times[0] - 16.0
print(f"blip: {blip.size} samples at {fs_blip:.0f} Hz, peak at GPS {blip_times[0] + 16.0:.3f}")

# band-pass 40 to 300 Hz, then a notch at 60 Hz, both forwards and backwards
b_g, a_g = signal.butter(4, [40.0 / (fs_blip / 2), 300.0 / (fs_blip / 2)], btype='band')
b_60, a_60, _ = notch_coefficients(60.0, 1.0, fs_blip)
clean = signal.filtfilt(b_60, a_60, signal.filtfilt(b_g, a_g, blip))

# the noise level away from the glitch, then the largest value near it
quiet = (np.abs(t_blip) > 1) & (np.abs(t_blip) < 10)
sigma = clean[quiet].std()
near = np.abs(t_blip) < 0.5
peak = np.abs(clean[near]).max() / sigma
print(f"sigma = {sigma:.2e}; largest value within 0.5 s: {peak:.1f} sigma, at {1e3 * t_blip[near][np.argmax(np.abs(clean[near]))]:.1f} ms")

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(t_blip[near], clean[near] / sigma, color=P['BLUE'], lw=0.7)
ax.set(xlabel='time from the peak of the blip (s)', ylabel=r'strain / $\sigma$')
plt.show()

**What to look at.** A spike of about fifteen $\sigma$ at zero, a few milliseconds long, in noise of one or two $\sigma$. In time we see when it is. We do not see its frequencies.

## 8 · The spectrogram: a short and a long window

**What we do.** The short-time Fourier transform cuts the record into windows of $N_{w}$ samples with a Hann window. Then it takes the DFT of each one. The squared modulus is the spectrogram. First we keep a wide band, 20 to 1000 Hz. Then each frequency row is divided by its median over the record: a value of 1 is the usual level at that frequency. We try $N_{w} = 256$ and $N_{w} = 2048$.

In [ ]:
def normalized_spectrogram(x, fs, n_w):
    """The spectrogram with a Hann window of n_w samples, each row divided by its median.

    Parameters
    ----------
    x   : array, the record
    fs  : float, sampling frequency (Hz)
    n_w : int, samples in one window

    Returns
    -------
    (freqs, t_mid, power) : the frequencies (Hz); the center of each window (s from the start); the power over its median
    """
    # windows that overlap by all but 64 samples, so the columns are 64 samples apart
    freqs, t_mid, sp = signal.spectrogram(x, fs=fs, window='hann', nperseg=n_w, noverlap=n_w - 64)
    power = sp / np.median(sp, axis=1, keepdims=True)
    result = (freqs, t_mid, power)
    return result


# a wide band for the maps: 20 to 1000 Hz
b_w, a_w = signal.butter(4, [20.0 / (fs_blip / 2), 1000.0 / (fs_blip / 2)], btype='band')
wide = signal.filtfilt(b_w, a_w, blip)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.4), sharey=True)
for axis, n_w in zip(ax, (256, 2048)):
    f_sp, t_sp, power = normalized_spectrogram(wide, fs_blip, n_w)
    t_sp = t_sp - 16.0
    cols = np.abs(t_sp) < 1
    rows = (f_sp >= 20) & (f_sp <= 1000)
    i, j = np.unravel_index(np.argmax(power[rows][:, cols]), power[rows][:, cols].shape)
    print(f"N_w = {n_w}: window {1e3 * n_w / fs_blip:.1f} ms, bins {fs_blip / n_w:.1f} Hz apart, product {(n_w / fs_blip) * (fs_blip / n_w):.0f}; "
          f"largest {power[rows][:, cols].max():.0f} at {f_sp[rows][i]:.0f} Hz, {1e3 * t_sp[cols][j]:.0f} ms")
    mesh = axis.pcolormesh(t_sp[cols], f_sp[rows], power[rows][:, cols], vmin=0, vmax=20, shading='auto', cmap='magma')
    axis.set(yscale='log', ylim=(20, 1000), xlabel='time from the peak of the blip (s)', title=f'$N_w$ = {n_w}')
    axis.set_yticks([20, 50, 100, 200, 500, 1000], labels=['20', '50', '100', '200', '500', '1000'])
    axis.yaxis.set_minor_formatter(plt.NullFormatter())
ax[0].set_ylabel('frequency (Hz)')
fig.colorbar(mesh, ax=ax, label='power / its median')
plt.show()

**What to look at.** With 256 samples the blip is a thin vertical stripe, from about 70 Hz to about 500 Hz: sharp in time, coarse in frequency. With 2048 samples it is a wide spot, a few tenths of a second long: fine in frequency, but the blip is much shorter than the window. The window length times the bin width is always 1. We only choose the shape of the tile.

## 9 · The Q transform: every tile with the same $Q$

**What we do.** A short feature at high frequency needs short tiles. A slow one at low frequency needs long tiles. The constant-Q transform keeps $Q$, the center frequency over the width, the same at every frequency. The width changes. We use `q_transform` of the course module `gwcourse_signal`, with $Q = 12$. It divides each row by its median, as in step 8, so it shows the energy above the usual level. We give it the wide band-passed record divided by its rms, so that the numbers are of order one.

In [ ]:
# the Q transform within 0.5 s of the peak, from 20 to 1000 Hz
t_q, f_q, energy = q_transform(wide / wide.std(), fs_blip, t_blip, 0.0, half=0.5, q=12.0, frange=(20.0, 1000.0))
i, j = np.unravel_index(np.argmax(energy), energy.shape)
print(f"largest energy {energy.max():.0f} at {f_q[i]:.0f} Hz, {1e3 * t_q[j]:.1f} ms")

fig, ax = plt.subplots(figsize=(10, 4.4))
mesh = ax.pcolormesh(t_q, f_q, energy, vmin=0, vmax=25, shading='auto', cmap='magma')
ax.set(yscale='log', xlabel='time from the peak of the blip (s)', ylabel='frequency (Hz)')
ax.set_yticks([20, 50, 100, 200, 500, 1000], labels=['20', '50', '100', '200', '500', '1000'])
ax.yaxis.set_minor_formatter(plt.NullFormatter())
fig.colorbar(mesh, ax=ax, label='energy / its median')
plt.show()

**What to look at.** A teardrop. It is wider in time at the bottom, near 80 Hz. It is thin at the top, up to about 700 Hz. At low frequency the tiles are long, at high frequency they are short. This is the kind of map of the glitch families in lecture 11: now we know how it is made.

## 10 · Wavelets: the Morlet scalogram

**What we do.** We compare the data with one short wave, stretched to the scale $a$ and shifted to the time $b$:

$$
W(a,b) = \int x(t)\,\psi^{*}_{a,b}(t)\,\mathrm{d}t, \qquad
\psi_{a,b}(t) = \frac{1}{\sqrt{a}}\,\psi\!\left(\frac{t-b}{a}\right), \qquad
\psi(t) = \pi^{-1/4}\,\mathrm{e}^{\mathrm{i}\omega_{0}t}\,\mathrm{e}^{-t^{2}/2}.
$$

The center frequency of the scale $a$ is $f_{0} = \omega_{0}/(2\pi a)$, so for each frequency we take $a = \omega_{0}/(2\pi f_{0})$. `signal.correlate` computes the integral for every shift $b$ at once. The scalogram is $|W|^{2}$; we divide each row by its median, as before. The lecture showed that $Q = \sqrt{2}\,\omega_{0}$ at every scale, with $\sigma_{t} = a/\sqrt{2}$ and $\sigma_{f} = 1/(2\sqrt{2}\,\pi a)$.

In [ ]:
def morlet_scalogram(x, fs, freqs, w0=6.0):
    """The Morlet scalogram |W(a,b)|^2, each row divided by its median.

    Parameters
    ----------
    x     : array, the record
    fs    : float, sampling frequency (Hz)
    freqs : array, the center frequencies f_0 of the rows (Hz)
    w0    : float, the Morlet parameter omega_0

    Returns
    -------
    power : array (len(freqs), len(x)), the scalogram over its median, row by row
    """
    power = np.empty((freqs.size, x.size))
    for row, f0 in enumerate(freqs):
        # the scale of this row and the wavelet on +-5 a, where the Gaussian has fallen to 4e-6
        a = w0 / (2 * np.pi * f0)
        t = np.arange(-5 * a, 5 * a, 1 / fs)
        psi = np.pi ** -0.25 * np.exp(1j * w0 * t / a) * np.exp(-(t / a) ** 2 / 2) / np.sqrt(a)
        # W(a, b) for every shift b: the correlation of the data with the wavelet, times dt
        w_ab = signal.correlate(x, psi, mode='same', method='fft') / fs
        power[row] = np.abs(w_ab) ** 2
    power = power / np.median(power, axis=1, keepdims=True)
    return power


# Q, the widths and their product, from the formulas of the lecture
w0 = 6.0
print(f"Q = sqrt(2) w0 = {np.sqrt(2) * w0:.2f}")
for f0 in (50.0, 300.0):
    a = w0 / (2 * np.pi * f0)
    sigma_t, sigma_f = a / np.sqrt(2), 1 / (2 * np.sqrt(2) * np.pi * a)
    print(f"f_0 = {f0:.0f} Hz: a = {1e3 * a:.2f} ms, sigma_t = {1e3 * sigma_t:.2f} ms, sigma_f = {sigma_f:.1f} Hz, product {sigma_t * sigma_f:.4f}")

# the scalogram of 8 s of the wide band-passed record, drawn within 0.5 s of the peak
seg = np.abs(t_blip) < 4
f_rows = np.geomspace(20.0, 1000.0, 80)
scal = morlet_scalogram(wide[seg], fs_blip, f_rows, w0)
cols = np.abs(t_blip[seg]) < 0.5
i, j = np.unravel_index(np.argmax(scal[:, cols]), scal[:, cols].shape)
print(f"largest {scal[:, cols].max():.0f} at {f_rows[i]:.0f} Hz, {1e3 * t_blip[seg][cols][j]:.1f} ms")

fig, ax = plt.subplots(figsize=(10, 4.4))
mesh = ax.pcolormesh(t_blip[seg][cols], f_rows, scal[:, cols], vmin=0, vmax=25, shading='auto', cmap='magma')
ax.set(yscale='log', xlabel='time from the peak of the blip (s)', ylabel='frequency (Hz)')
ax.set_yticks([20, 50, 100, 200, 500, 1000], labels=['20', '50', '100', '200', '500', '1000'])
ax.yaxis.set_minor_formatter(plt.NullFormatter())
fig.colorbar(mesh, ax=ax, label='$|W|^2$ / its median')
plt.show()

**What to look at.** $Q$ is the same at 50 and at 300 Hz. The tile is six times shorter at 300 Hz and six times wider in frequency. The product $\sigma_{t}\sigma_{f}$ is $1/4\pi = 0.0796$ at both. The scalogram shows the same drop as the Q transform: the Morlet wavelet is one more constant-Q tiling of the plane.

## What we saw

- The record keeps nothing above $f_{N}$. Keeping one sample in $M$ without a low-pass makes a violin mode appear at 66.75 Hz.
- A digital filter is fixed by its coefficients. We check its poles and its gain before we use it. Forwards and backwards it gives zero phase and the gain twice.
- A band-pass removes the wall and the violin modes. A notch removes one line and leaves its neighbors.
- A glitch is clear in time. The spectrogram, the Q transform and the wavelet show when it happens and at which frequencies.

At home, `H05` repeats these steps on a glitch in LIGO Livingston.

## Software

`numpy`, `scipy.signal` (`welch`, `butter`, `freqz`, `filtfilt`, `spectrogram`, `correlate`) and `matplotlib` (Virtanen et al. 2020). The Q transform is `q_transform` of the course module `gwcourse_signal`, after (Chatterji et al. 2004). The Morlet wavelet with $\omega_{0} = 6$ follows (Torrence and Compo 1998). The data come from GWOSC (LIGO Scientific Collaboration and Virgo Collaboration 2021), (LIGO Scientific, Virgo and KAGRA Collaborations 2023), downloaded with `gwpy` (`TimeSeries.fetch_open_data`); the class of the glitch comes from Gravity Spy (Zevin et al. 2017).

**References**

- Orfanidis (1996). *Introduction to Signal Processing*.
- Harris (1978). *On the use of windows for harmonic analysis with the discrete Fourier transform*.
- Chatterji et al. (2004). *Multiresolution techniques for the detection of gravitational-wave bursts*.
- Torrence and Compo (1998). *A Practical Guide to Wavelet Analysis*.
- LIGO Scientific and Virgo Collaborations (2020). *GW190814: Gravitational Waves from the Coalescence of a 23 Solar Mass Black Hole with a 2.6 Solar Mass Compact Object*.
- Zevin et al. (2017). *Gravity Spy: integrating advanced LIGO detector characterization, machine learning, and citizen science*.
- LIGO Scientific Collaboration and Virgo Collaboration (2021). *Open data from the first and second observing runs of Advanced LIGO and Advanced Virgo*.
- LIGO Scientific, Virgo and KAGRA Collaborations (2023). *Open Data from the Third Observing Run of LIGO, Virgo, KAGRA, and GEO*.
- Virtanen et al. (2020). *SciPy 1.0: fundamental algorithms for scientific computing in Python*.